In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)


# ==========================
# OpenAI Configuration
# ==========================

config_str = os.getenv("GPT_LUNA_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)

MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)

DATA_PATH = os.getenv("DATA_PATH", "../data/")

df=pd.read_csv(f'{DATA_PATH}df_n3.csv')



Using model: gpt-5.6-luna with reasoning effort: low


In [3]:
import json
import random
from collections import defaultdict, Counter

import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

# Your original dataframe
# Example:
# df = pd.read_pickle("my_original_df.pkl")
#
# If df is already loaded in your notebook, you don't need
# to change anything here.

BATCH_RESULTS_PATH = "dedup_batch_results_50.jsonl"

N_CONSISTENCY_RUNS = 3


# ============================================================
# 1. LOAD BATCH RESULTS
# ============================================================

def load_batch_results(jsonl_path):
    """
    Load OpenAI batch results from a JSONL file.

    Returns:
        list of raw batch result objects
    """

    results = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            results.append(json.loads(line))

    print(f"[BATCH] Loaded {len(results)} batch results")

    return results


# ============================================================
# 2. EXTRACT LLM RESPONSE FROM BATCH RESULT
# ============================================================

def extract_batch_response(batch_result):
    """
    Extract the text returned by the LLM from a batch result.

    Expected OpenAI Batch structure:

    {
        "custom_id": "...",
        "response": {
            "status_code": 200,
            "body": {
                "choices": [
                    {
                        "message": {
                            "content": "..."
                        }
                    }
                ]
            }
        }
    }
    """

    try:
        response = batch_result["response"]

        body = response["body"]

        choices = body["choices"]

        if not choices:
            return None

        return choices[0]["message"]["content"]

    except Exception as e:
        print(f"[WARN] Could not extract response: {e}")
        return None


# ============================================================
# 3. GET CUSTOM ID
# ============================================================

def get_custom_id(batch_result):
    """
    Return the custom_id used when creating the batch request.

    Change this function ONLY if your batch custom_id has a
    particular structure.
    """

    return batch_result.get("custom_id")


# ============================================================
# 4. PARSE + VALIDATE LLM OUTPUT
# ============================================================

def parse_dedup_output(text, valid_ids):
    """
    Same logic as the original implementation.

    - Removes hallucinated IDs
    - Removes duplicated IDs
    - Adds forgotten IDs as singleton groups
    - Removes synthesis_comment from singleton groups
    """

    if text is None:
        return None

    # Sometimes models return ```json ... ```
    # even though JSON only was requested.
    text = text.strip()

    if text.startswith("```"):
        text = text.replace("```json", "", 1)
        text = text.replace("```", "")
        text = text.strip()

    try:
        data = json.loads(text)
        raw_groups = data.get("groups", [])

    except Exception:
        return None

    valid_ids = set(valid_ids)

    seen = set()
    groups = []

    for g in raw_groups:

        ids = g.get("comment_ids", [])

        clean_ids = [
            i
            for i in ids
            if i in valid_ids and i not in seen
        ]

        if not clean_ids:
            continue

        seen.update(clean_ids)

        synthesis = (
            g.get("synthesis_comment")
            if len(clean_ids) > 1
            else None
        )

        if isinstance(synthesis, str):
            synthesis = synthesis.strip() or None

        groups.append(
            {
                "comment_ids": clean_ids,
                "synthesis_comment": synthesis,
            }
        )

    # Add comments that the LLM forgot
    missing = valid_ids - seen

    for m in missing:
        groups.append(
            {
                "comment_ids": [m],
                "synthesis_comment": None,
            }
        )

    return groups


# ============================================================
# 5. GROUPS -> PAIRS
# ============================================================

def groups_to_pairs(groups):
    """
    Convert groups into pairs of comments that co-occurred
    in the same group.
    """

    pairs = set()

    for g in groups:

        ids = g["comment_ids"]

        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):

                pairs.add(
                    frozenset(
                        (ids[i], ids[j])
                    )
                )

    return pairs


# ============================================================
# 6. UNION-FIND
# ============================================================

class UnionFind:

    def __init__(self, ids):

        self.parent = {
            i: i
            for i in ids
        }

    def find(self, x):

        while self.parent[x] != x:

            self.parent[x] = self.parent[
                self.parent[x]
            ]

            x = self.parent[x]

        return x

    def union(self, a, b):

        ra = self.find(a)
        rb = self.find(b)

        if ra != rb:
            self.parent[ra] = rb


# ============================================================
# 7. SELECT SYNTHESIS COMMENT
# ============================================================

def select_synthesis_for_cluster(
    cluster_ids,
    all_run_groups
):

    target = set(cluster_ids)

    best_group = None
    best_score = -1.0

    for g in all_run_groups:

        if not g.get("synthesis_comment"):
            continue

        gid_set = set(
            g["comment_ids"]
        )

        # Exact match = best possible
        if gid_set == target:

            return g["synthesis_comment"]

        intersection = len(
            gid_set & target
        )

        if intersection == 0:
            continue

        union_size = len(
            gid_set | target
        )

        jaccard = (
            intersection / union_size
        )

        if jaccard > best_score:

            best_score = jaccard
            best_group = g

    return (
        best_group["synthesis_comment"]
        if best_group
        else None
    )


# ============================================================
# 8. AGREEMENT SCORES
# ============================================================

def get_cluster_agreement(
    cluster_ids,
    pair_agreements
):

    agreements = []

    for i in range(len(cluster_ids)):

        for j in range(i + 1, len(cluster_ids)):

            pair = frozenset(
                [
                    cluster_ids[i],
                    cluster_ids[j]
                ]
            )

            if pair in pair_agreements:

                agreements.append(
                    pair_agreements[pair][
                        "agreement_ratio"
                    ]
                )

    return agreements


# ============================================================
# 9. BUILD FINAL CLUSTER ROWS
# ============================================================

def build_cluster_rows(
    patch_id,
    clusters,
    rows,
    all_run_groups,
    pair_agreements
):

    lookup = {
        r["comment_id"]: r
        for r in rows
    }

    cluster_rows = []

    for k, ids in enumerate(clusters):

        synthesis = (
            select_synthesis_for_cluster(
                ids,
                all_run_groups
            )
            if len(ids) > 1
            else None
        )

        cluster_rows.append(
            {
                "patch_id": patch_id,

                "num_comments": len(ids),

                "cluster_id":
                    f"{patch_id}_c{k}",

                "comment_ids": ids,

                "comments": [
                    lookup[i]["generated_comment"]
                    for i in ids
                ],

                "generation_systems": [
                    lookup[i]["generation_system"]
                    for i in ids
                ],

                "categories": [
                    lookup[i]["category"]
                    for i in ids
                ],

                "severities": [
                    lookup[i]["severity"]
                    for i in ids
                ],

                "synthesis_comment":
                    synthesis,

                "agreement_scores":
                    get_cluster_agreement(
                        ids,
                        pair_agreements
                    ),
            }
        )

    return cluster_rows


# ============================================================
# 10. PROCESS ONE PATCH FROM BATCH RESULTS
# ============================================================

def process_patch_from_batch(
    patch_id,
    rows,
    run_outputs
):
    """
    Reproduces the post-LLM processing of the original
    run_dedup_for_patch(), but WITHOUT making any API call.

    run_outputs must contain one LLM response per
    consistency run.
    """

    valid_ids = [
        r["comment_id"]
        for r in rows
    ]

    # --------------------------------------------------------
    # Singleton patch
    # --------------------------------------------------------

    if len(rows) <= 1:

        clusters = [
            [r["comment_id"]]
            for r in rows
        ]

        return build_cluster_rows(
            patch_id,
            clusters,
            rows,
            all_run_groups=[],
            pair_agreements={}
        )

    # --------------------------------------------------------
    # Process the 3 already-generated LLM outputs
    # --------------------------------------------------------

    run_pair_sets = []

    all_run_groups = []

    for run_idx, raw_text in enumerate(run_outputs):

        groups = parse_dedup_output(
            raw_text,
            valid_ids
        )

        if groups is None:

            print(
                f"[WARN] patch {patch_id} "
                f"run {run_idx}: "
                f"unparseable output"
            )

            groups = [
                {
                    "comment_ids": [i],
                    "synthesis_comment": None
                }
                for i in valid_ids
            ]

        run_pair_sets.append(
            groups_to_pairs(groups)
        )

        all_run_groups.extend(groups)

    # --------------------------------------------------------
    # Majority agreement
    # --------------------------------------------------------

    min_votes = (
        len(run_pair_sets) // 2 + 1
    )

    pair_counts = Counter()

    for pairs in run_pair_sets:

        for pair in pairs:

            pair_counts[pair] += 1

    pair_agreements = {

        pair: {
            "agreement_count": count,
            "agreement_ratio":
                count / len(run_pair_sets)
        }

        for pair, count
        in pair_counts.items()
    }

    # --------------------------------------------------------
    # Keep majority-agreed pairs
    # --------------------------------------------------------

    consistent_pairs = {

        pair

        for pair, info
        in pair_agreements.items()

        if info["agreement_count"]
        >= min_votes
    }

    # --------------------------------------------------------
    # Union-Find
    # --------------------------------------------------------

    uf = UnionFind(valid_ids)

    for pair in consistent_pairs:

        a, b = tuple(pair)

        uf.union(a, b)

    # --------------------------------------------------------
    # Build clusters
    # --------------------------------------------------------

    clusters_map = defaultdict(list)

    for comment_id in valid_ids:

        clusters_map[
            uf.find(comment_id)
        ].append(comment_id)

    clusters = list(
        clusters_map.values()
    )

    # --------------------------------------------------------
    # Final cluster rows
    # --------------------------------------------------------

    return build_cluster_rows(
        patch_id,
        clusters,
        rows,
        all_run_groups,
        pair_agreements
    )


# ============================================================
# 11. MAP BATCH RESULTS TO PATCHES / RUNS
# ============================================================

def build_batch_output_map(batch_results):
    """
    Converts the JSONL batch results into:

        {
            custom_id: llm_output
        }

    """

    output_map = {}

    for result in batch_results:

        custom_id = get_custom_id(
            result
        )

        if custom_id is None:
            print(
                "[WARN] Batch result has no custom_id"
            )
            continue

        output_map[
            custom_id
        ] = extract_batch_response(
            result
        )

    return output_map


# ============================================================
# 12. CUSTOM_ID -> PATCH + RUN
# ============================================================


# ============================================================
# 13. MAIN PIPELINE
# ============================================================
def run_batch_dedup_pipeline(
    df,
    batch_results_path
):
    """
    Post-process already completed batch results.

    The JSONL is assumed to be ordered as:

        patch 1 -> run 0
        patch 1 -> run 1
        patch 1 -> run 2

        patch 2 -> run 0
        patch 2 -> run 1
        patch 2 -> run 2

        ...

    No LLM/API calls are made here.
    """

    required_cols = [
        "comment_id",
        "generated_comment",
        "category",
        "generation_system",
        "patch_id",
        "severity",
        "hunk"
    ]

    for c in required_cols:

        if c not in df.columns:

            raise ValueError(
                f"Missing required column: {c}"
            )

    # --------------------------------------------------------
    # Load batch results
    # --------------------------------------------------------

    batch_results = load_batch_results(
        batch_results_path
    )

    batch_outputs = [
        extract_batch_response(result)
        for result in batch_results
    ]

    print(
        f"[BATCH] Loaded {len(batch_outputs)} outputs"
    )

    # --------------------------------------------------------
    # Group dataframe by patch
    # --------------------------------------------------------

    patch_groups = list(
        df.groupby("patch_id")
    )

    total_patches = len(patch_groups)

    expected_outputs = (
        total_patches *
        N_CONSISTENCY_RUNS
    )

    print(
        f"[INFO] Number of patches: "
        f"{total_patches}"
    )

    print(
        f"[INFO] Expected batch outputs: "
        f"{expected_outputs}"
    )

    print(
        f"[INFO] Actual batch outputs: "
        f"{len(batch_outputs)}"
    )

    if len(batch_outputs) != expected_outputs:

        raise ValueError(
            f"Number of batch results does not match "
            f"the expected number.\n"
            f"Expected: {expected_outputs} "
            f"({total_patches} patches × "
            f"{N_CONSISTENCY_RUNS} runs)\n"
            f"Found: {len(batch_outputs)}"
        )

    # --------------------------------------------------------
    # Process patches
    # --------------------------------------------------------

    results = []

    for patch_idx, (patch_id, group) in enumerate(
        patch_groups
    ):

        print(
            f"\n[PATCH {patch_idx + 1}/"
            f"{total_patches}] "
            f"{patch_id} "
            f"({len(group)} comments)"
        )

        rows = group.to_dict(
            "records"
        )

        for r in rows:

            r.setdefault(
                "generated_context",
                ""
            )

        # ----------------------------------------------------
        # Extract the 3 outputs corresponding to this patch
        # ----------------------------------------------------

        start_idx = (
            patch_idx *
            N_CONSISTENCY_RUNS
        )

        end_idx = (
            start_idx +
            N_CONSISTENCY_RUNS
        )

        run_outputs = batch_outputs[
            start_idx:end_idx
        ]

        print(
            f"  Using batch outputs "
            f"{start_idx} → {end_idx - 1}"
        )

        # ----------------------------------------------------
        # Process without API call
        # ----------------------------------------------------

        try:

            patch_cluster_rows = (
                process_patch_from_batch(
                    patch_id,
                    rows,
                    run_outputs
                )
            )

        except Exception as e:

            print(
                f"[FAIL] patch {patch_id}: {e}"
            )

            patch_cluster_rows = [

                {
                    "patch_id":
                        patch_id,

                    "num_comments":
                        len(rows),

                    "cluster_id":
                        f"{patch_id}_c{k}",

                    "comment_ids":
                        [r["comment_id"]],

                    "comments":
                        [r["generated_comment"]],

                    "generation_systems":
                        [r["generation_system"]],

                    "categories":
                        [r["category"]],

                    "severities":
                        [r["severity"]],

                    "synthesis_comment":
                        None,

                    "error":
                        str(e),
                }

                for k, r in enumerate(rows)
            ]

        results.extend(
            patch_cluster_rows
        )

    # --------------------------------------------------------
    # Final DataFrame
    # --------------------------------------------------------

    final_df = pd.DataFrame(
        results
    )

    print(
        "\n[DONE] Batch post-processing finished"
    )

    print(
        f"[DONE] Final dataframe shape: "
        f"{final_df.shape}"
    )

    return final_df

# ============================================================
# 14. RUN
# ============================================================

# Your original dataframe should already be loaded as `df`.
final_df = run_batch_dedup_pipeline(
    df=df,
    batch_results_path=BATCH_RESULTS_PATH
)

print(final_df.head())

[BATCH] Loaded 150 batch results
[BATCH] Loaded 150 outputs
[INFO] Number of patches: 50
[INFO] Expected batch outputs: 150
[INFO] Actual batch outputs: 150

[PATCH 1/50] P000000 (1 comments)
  Using batch outputs 0 → 2

[PATCH 2/50] P000001 (1 comments)
  Using batch outputs 3 → 5

[PATCH 3/50] P000002 (2 comments)
  Using batch outputs 6 → 8

[PATCH 4/50] P000003 (1 comments)
  Using batch outputs 9 → 11

[PATCH 5/50] P000004 (1 comments)
  Using batch outputs 12 → 14

[PATCH 6/50] P000005 (1 comments)
  Using batch outputs 15 → 17

[PATCH 7/50] P000006 (2 comments)
  Using batch outputs 18 → 20

[PATCH 8/50] P000007 (2 comments)
  Using batch outputs 21 → 23

[PATCH 9/50] P000008 (3 comments)
  Using batch outputs 24 → 26

[PATCH 10/50] P000009 (1 comments)
  Using batch outputs 27 → 29

[PATCH 11/50] P000010 (1 comments)
  Using batch outputs 30 → 32

[PATCH 12/50] P000011 (3 comments)
  Using batch outputs 33 → 35

[PATCH 13/50] P000012 (3 comments)
  Using batch outputs 36 → 38



In [11]:
final_df.head(50)

,patch_id,num_comments,cluster_id,comment_ids,comments,generation_systems,categories,severities,synthesis_comment,agreement_scores
0,P000000,1,P000000_c0,[e582ad72-8d9f-418d-a017-4c605b4a6009],[The docstring for the `head` method mentions ...,[gpt-4o-mini-2024-07-18],[Documentation],[Medium],None,[]
1,P000001,1,P000001_c0,[557c8ad0-c25c-4c31-bdb5-463aa09f6dd2],[The `cross_entropy` function call is replaced...,[gpt-4o-mini-2024-07-18],[Correctness],[High],None,[]
2,P000002,2,P000002_c0,"[d1219982-9fea-45e9-97a0-65aeae62d25c, 7265ac7...",[Could we remove the remaining `stylesheet.set...,"[gpt-5.4-mini-2026-03-17, gpt-4o-mini-2024-07-18]","[Maintainability, Correctness]","[Low, Medium]",Now that the line edit's stylesheet is defined...,[1.0]
3,P000003,1,P000003_c0,[cda853dd-b7df-47e2-a9d3-a2d75a5d3b05],[The docstring for `iter_kubernetes_nodes` men...,[gpt-4o-mini-2024-07-18],[Documentation],[Medium],None,[]
4,P000004,1,P000004_c0,[d35ae336-9da1-4d4f-80ce-f2a73dced7d4],[The docstring for `naive_dice_loss` mentions ...,[gpt-4o-mini-2024-07-18],[Documentation],[Medium],None,[]
5,P000005,1,P000005_c0,[7506dca5-ec7c-4dd0-afd3-59fcf8c4b358],[Is it intentional to remove the deprecation w...,[gpt-4o-mini-2024-07-18],[Correctness],[Medium],None,[]
6,P000006,2,P000006_c0,"[9c09de10-d775-42dc-998c-f75205f8a305, 6f13e3b...",[The method call to `_compact_times()` has bee...,"[gpt-4o-mini-2024-07-18, gpt-5.4-mini-2026-03-17]","[Correctness, Correctness]","[High, Medium]",Removing response-time compaction from recalcu...,[1.0]
7,P000007,1,P000007_c0,[3e75fbd5-661c-48e0-9d7c-918374329169],[The call to `utils.to_native_string` is now p...,[gpt-4o-mini-2024-07-18],[Correctness],[High],None,[]
8,P000007,1,P000007_c1,[cf3a5d04-5f50-4454-8d54-ff8fc104374b],[Could this preserve the type of the parsed UR...,[gpt-5.6-luna],[Correctness],[Medium],None,[]
9,P000008,2,P000008_c0,"[864abeeb-9771-46f1-9308-2dd004c8042a, a4f8c28...","[Could this retain the `getattr(cls, ""task_nam...","[gpt-5.6-luna, gpt-5.4-mini-2026-03-17]","[Correctness, Correctness]","[Medium, Medium]",Directly accessing `cls.task_namespace` can ra...,[1.0]


In [4]:
import pandas as pd

jsonl_path = "dedup_batch_results_50.jsonl"

batch_df = pd.read_json(
    jsonl_path,
    lines=True
)

batch_df.head()

,id,custom_id,response,error
0,batch_req_6a82f2065e50819097c30378e3921406,dedup_P000000_run_0,"{'status_code': 200, 'request_id': '201d5513-c...",NaN
1,batch_req_6a82f2067bf48190b712a6b1b7802c88,dedup_P000000_run_1,"{'status_code': 200, 'request_id': '35f32c1f-f...",NaN
2,batch_req_6a82f2067c54819099993a6146994d08,dedup_P000000_run_2,"{'status_code': 200, 'request_id': '8d18de7d-a...",NaN
3,batch_req_6a82f20676a88190a499b595c5761e0a,dedup_P000001_run_0,"{'status_code': 200, 'request_id': '5703cc9e-0...",NaN
4,batch_req_6a82f20675208190a4586b254fbee522,dedup_P000001_run_1,"{'status_code': 200, 'request_id': '36fab436-0...",NaN


In [ ]:
df = pd.read_csv(f'{DATA_PATH}df_n0.csv')

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   old_hunk  100 non-null    object
 1   oldf      100 non-null    object
 2   hunk      100 non-null    object
 3   comment   100 non-null    object
 4   ids       100 non-null    object
 5   repo      100 non-null    object
 6   ghid      100 non-null    int64 
 7   old       100 non-null    object
 8   new       100 non-null    object
 9   lang      100 non-null    object
 10  patch_id  100 non-null    object
dtypes: int64(1), object(10)
memory usage: 8.7+ KB


In [8]:
df_comments = pd.read_csv(f'{DATA_PATH}df_n3.csv')

In [10]:
df_comments.head(50)

,comment_id,generated_comment,category,generation_system,patch_id,severity,generation_file,human_comment,hunk,relevant_context,relevant_same_file_code_hunks,relevant_different_files_code_hunks
0,e582ad72-8d9f-418d-a017-4c605b4a6009,The docstring for the `head` method mentions t...,Documentation,gpt-4o-mini-2024-07-18,P000000,Medium,df_n3_3.csv,"Is the name ""head"" a convention for health che...","@@ -274,6 +274,7 @@ class RootPathHandler(Base...",class RootPathHandler(BaseTaskHistoryHandler):...,"[{'filename': 'luigi/server.py', 'patch': '@@ ...","[{'filename': 'test/server_test.py', 'patch': ..."
1,557c8ad0-c25c-4c31-bdb5-463aa09f6dd2,The `cross_entropy` function call is replaced ...,Correctness,gpt-4o-mini-2024-07-18,P000001,High,df_n3_3.csv,"`F.cross_entropy`? Also, isn't `tran_acc` requ...","@@ -117,7 +117,8 @@ def main(args):\n ...",def main(args):\n # split dataset into trai...,[{'filename': 'examples/mxnet/rgcn/entity_clas...,[]
2,d1219982-9fea-45e9-97a0-65aeae62d25c,Could we remove the remaining `stylesheet.set_...,Maintainability,gpt-5.4-mini-2026-03-17,P000002,Low,df_n3_1.csv,Couldn't you move the stylesheet to `ConsoleWi...,"@@ -48,12 +48,6 @@ class ConsoleLineEdit(miscw...",class ConsoleLineEdit(miscwidgets.CommandLineE...,[{'filename': 'qutebrowser/misc/consolewidget....,"[{'filename': 'doc/help/settings.asciidoc', 'p..."
3,7265ac74-5a5c-4090-be08-b00cff51e8f5,The `STYLESHEET` variable has been removed wit...,Correctness,gpt-4o-mini-2024-07-18,P000002,Medium,df_n3_3.csv,Couldn't you move the stylesheet to `ConsoleWi...,"@@ -48,12 +48,6 @@ class ConsoleLineEdit(miscw...",class ConsoleLineEdit(miscwidgets.CommandLineE...,[{'filename': 'qutebrowser/misc/consolewidget....,"[{'filename': 'doc/help/settings.asciidoc', 'p..."
4,cda853dd-b7df-47e2-a9d3-a2d75a5d3b05,The docstring for `iter_kubernetes_nodes` ment...,Documentation,gpt-4o-mini-2024-07-18,P000003,Medium,df_n3_3.csv,please update all the docstring below in other...,"@@ -2229,7 +2229,7 @@ class ApiClientImpl(ApiC...",class ApiClientImpl(ApiClient):\n def iter_...,[{'filename': 'google/cloud/forseti/services/i...,[{'filename': 'google/cloud/forseti/services/i...
5,d35ae336-9da1-4d4f-80ce-f2a73dced7d4,The docstring for `naive_dice_loss` mentions t...,Documentation,gpt-4o-mini-2024-07-18,P000004,Medium,df_n3_3.csv,use_second_power=True -> naive_dice=False,"@@ -51,8 +51,8 @@ def naive_dice_loss(pred,\n ...","def naive_dice_loss(pred,\n ...",[{'filename': 'mmdet/models/losses/dice_loss.p...,[{'filename': 'configs/maskformer/maskformer_r...
6,7506dca5-ec7c-4dd0-afd3-59fcf8c4b358,Is it intentional to remove the deprecation wa...,Correctness,gpt-4o-mini-2024-07-18,P000005,Medium,df_n3_3.csv,```suggestion ``` I can't find the versionchan...,"@@ -903,10 +903,6 @@ class PDBWriter(base.Writ...",class PDBWriter(base.WriterBase):\n def wri...,[],"[{'filename': 'package/CHANGELOG', 'patch': ""@..."
7,9c09de10-d775-42dc-998c-f75205f8a305,The method call to `_compact_times()` has been...,Correctness,gpt-4o-mini-2024-07-18,P000006,High,df_n3_3.csv,We don't need to compact times for each KPIset...,"@@ -167,8 +167,6 @@ class KPISet(BetterDict):\...",class KPISet(BetterDict):\n def __init__(se...,"[{'filename': 'bzt/modules/aggregator.py', 'pa...","[{'filename': 'site/dat/docs/Changelog.md', 'p..."
8,6f13e3b2-29b3-4493-ac9f-96481d60f39a,`recalculate()` no longer compacts `RESP_TIMES...,Correctness,gpt-5.4-mini-2026-03-17,P000006,Medium,df_n3_1.csv,We don't need to compact times for each KPIset...,"@@ -167,8 +167,6 @@ class KPISet(BetterDict):\...",class KPISet(BetterDict):\n def __init__(se...,"[{'filename': 'bzt/modules/aggregator.py', 'pa...","[{'filename': 'site/dat/docs/Changelog.md', 'p..."
9,3e75fbd5-661c-48e0-9d7c-918374329169,The call to `utils.to_native_string` is now pr...,Correctness,gpt-4o-mini-2024-07-18,P000007,High,df_n3_3.csv,A quick dive into the CPython codebase shows t...,"@@ -55,14 +55,8 @@ class MockRequest(object):\...","from .compat import urlparse, urlunpars